# Kompresi MuQ: INT8 vs palette6

Kedua kandidat dibuat langsung dari MuQ FP32 yang telah lolos full validation. MusicFM palette6 dan head palette6 tetap. Evaluasi menghitung MuQ saja, memakai ulang fitur MusicFM; tidak mengubah dataset atau aset SongMap.

Kernel `.venv-songformer311`, macOS, coremltools 8.3.0. Sediakan beberapa GiB disk untuk paket dan dua cache tambahan. Tutup training atau inferensi lain pada Mac 24 GB. Jalankan berurutan; proses kompresi palette6 dapat memerlukan waktu.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess, sys
assert Path('finetuning/config.json').is_file()
SONGMAP_ROOT=Path('/Users/danendra/Documents/ADA26/C6/SongMap')
reports=sorted(Path('outputs.local').glob('muq-validation-*/all-validation.json'),key=lambda p:p.stat().st_mtime,reverse=True)
assert reports, 'Full validation MuQ FP32 belum ditemukan'
validation=reports[0]
v=json.loads(validation.read_text())
assert v['full_validation'] and v['muq_scheme']=='fp32'
source=Path(v['muq_export_report'])
baseline=Path(v['baseline_combination'])
output=Path('outputs.local')/('muq-compression-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
print('Python:',sys.executable)
print('FP32 validation:',validation)
print('Output:',output)
print('FP32 F1:',v['metrics']['macro_f1_present'])


In [ ]:
def run_logged(command,log):
    log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Proses gagal ({status}); lihat {log.resolve()}')

def compress(scheme):
    folder=output/scheme
    if (folder/'compression_report.json').is_file():
        print('Kandidat sudah tersedia:',folder)
        return
    # Jika proses kompresi pernah gagal setelah membuat folder, pilih folder output baru.
    run_logged([sys.executable,'-u','-m','deployment.compress_muq',
        '--source-report',str(source),'--validation-report',str(validation),
        '--scheme',scheme,'--output-dir',str(folder)],output/(scheme+'-compression.log'))

def evaluate(scheme,limit,name):
    report=output/(scheme+'-'+name+'.json')
    if report.exists():
        print('Report sudah tersedia:',report)
        return
    run_logged([sys.executable,'-u','-m','deployment.evaluate_muq',
        '--songmap-root',str(SONGMAP_ROOT),'--baseline-combination',str(baseline),
        '--muq-export-report',str(output/scheme/'compression_report.json'),
        '--cache-dir',str(output/(scheme+'-cache')),'--limit',str(limit),
        '--output',str(report)],report.with_suffix('.log'))


In [ ]:
compress('int8')
evaluate('int8',1,'one-validation')
evaluate('int8',0,'all-validation')


In [ ]:
compress('palette6')
evaluate('palette6',1,'one-validation')
evaluate('palette6',0,'all-validation')


In [ ]:
summary={'source_validation_report':str(validation),'source_fp32_metrics':v['metrics'],'candidates':[],'ready_for_app_integration':False}
for scheme in ('int8','palette6'):
    c=json.loads((output/scheme/'compression_report.json').read_text())
    r=json.loads((output/(scheme+'-all-validation.json')).read_text())
    assert r['full_validation']
    row={'scheme':scheme,'size_mib':c['candidate_bytes']/1024**2,
         'size_reduction_fraction':c['size_reduction_fraction'],
         'macro_f1':r['metrics']['macro_f1_present'],
         'f1_delta_from_muq_fp32':r['macro_f1_delta_from_source_fp32'],
         'section_accuracy':r['metrics']['section_accuracy'],
         'low_coverage_sections':r['metrics']['low_coverage_sections'],
         'validation_report':str(output/(scheme+'-all-validation.json'))}
    summary['candidates'].append(row)
    print(json.dumps(row,indent=2))
    print('Per-class:',r['metrics']['per_class'])
(output/'comparison.json').write_text(json.dumps(summary,indent=2)+'\n')
print('Bagikan:',output/'comparison.json')
print('Belum mengukur latency/memori iPhone. Weight compression tidak menjamin integer compute atau speedup.')
